In [1]:
import numpy as np
import nbimporter
import cepairsimplementation as ce

In [2]:
def gaussian_kernel(X, Y, sigma):
    """Compute the Gaussian kernel matrix between X and Y."""
    sq_dists = np.sum(X**2, axis=1, keepdims=True) + np.sum(Y**2, axis=1) - 2 * X @ Y.T
    K = np.exp(-sq_dists / (2 * sigma**2))
    return K

def emd(d,lambda_=0.1):
    x,y=d
    if x.shape[1]>1 or y.shape[1]>1:
        return np.nan
    x=ce.minmax_scale(x)
    y=ce.minmax_scale(y)

    L = x.shape[0]

    ux = np.random.rand(L, x.shape[1])
    uy = np.random.rand(L, y.shape[1])

    Kx=gaussian_kernel(x, x, 1)
    Ky=gaussian_kernel(y, y, 1)
    Kxux=gaussian_kernel(x, ux, 1)
    Kyuy=gaussian_kernel(y, uy, 1)
    Kux=gaussian_kernel(ux, ux, 1)
    Kuy=gaussian_kernel(uy, uy, 1)

    Vx = np.linalg.inv(Kx + np.eye(L) * lambda_ * L)
    Vy = np.linalg.inv(Ky + np.eye(L) * lambda_ * L)
        
    # Cxy terms
    first = (1 / L**2) * np.trace(Kx @ Ky) \
            - (2 / L**2) * np.trace(Ky @ Vx @ Kxux @ Kxux.T) \
            + (1 / L**2) * np.trace(Ky @ Vx @ Kxux @ Kux @ Kxux.T @ Vx)

    second = (1 / L**2) * np.trace(Ky @ Vx @ Kxux @ Kux @ Kxux.T @ Vx) \
             - (2 / L**2) * np.trace(Kyuy.T @ Vx @ Kxux @ Kux) \
             + (1 / L**2) * np.trace(Kux @ Kuy)

    # Cyx terms
    third = (1 / L**2) * np.trace(Ky @ Kx) \
            - (2 / L**2) * np.trace(Kx @ Vy @ Kyuy @ Kyuy.T) \
            + (1 / L**2) * np.trace(Kx @ Vy @ Kyuy @ Kuy @ Kyuy.T @ Vy)

    fourth = (1 / L**2) * np.trace(Kx @ Vy @ Kyuy @ Kuy @ Kyuy.T @ Vy) \
             - (2 / L**2) * np.trace(Kxux.T @ Vy @ Kyuy @ Kuy) \
             + (1 / L**2) * np.trace(Kuy @ Kux)

    Cxy = first + second
    Cyx = third + fourth
    return Cyx - Cxy


In [3]:
print(ce.test_tuebingen(emd))

(np.float64(0.726504454822172), np.float64(0.6453546816839358))
